# LOGAN: candidate evidence by length, evidence class and biome

**Draft caption.** (a) Full-length structural hit rate by ORF length for candidates, complete Prodigal genes and the low-score intergenic and antisense controls (the controls lie together near zero). (b) Full-length evidence classes over all five databases: sequence+structure, structure only, sequence only, neither. (c) Full-length structural hit rate by biome for candidates and complete Prodigal genes. (d) Superkingdom of the best full-length target for candidates with a hit, by biome; shares reflect the composition of the target databases and are compared between arms rather than read as absolute.

**Outputs:** `../images/figL07_candidate_evidence_detail.pdf` (vector) and `../images/figL07_candidate_evidence_detail.png` (300 dpi).

**Data** (plotting-ready, committed; produced by `../prep/logan_prepare_data.py` on Setonix):

- `../data/figL07_evidence_by_length.tsv.gz` (16 rows; sha256 `836d04fa3fd7…`): full-length structural rate by arm and length bin. Source on Setonix (`$LOGAN_ROOT`): `qc/candidates/evidence_by_length.tsv`.
- `../data/figL07_evidence_classes.tsv.gz` (4 rows; sha256 `0dc7efa493c8…`): share of queries in each full-length evidence class. Source on Setonix (`$LOGAN_ROOT`): `qc/candidates/evidence_classes.tsv`.
- `../data/figL07_hit_biome.tsv.gz` (52 rows; sha256 `50d6cab79ff7…`): full-length structural hit rate and best-target superkingdom by arm and biome. Source on Setonix (`$LOGAN_ROOT`): `qc/candidates/hit_biome.tsv`.

**Upstream.** LOGAN v1.2 metagenomes (50,009), genome_entropy #102 (closed by PR #115); analysis scripts `Validation/logan/scripts/` at commit `06080b4`. Full run provenance: `../data/logan_provenance.json`.

**Run:** `jupyter nbconvert --to notebook --execute figL07_candidate_evidence_detail.ipynb` from `manuscript/notebooks/`. Needs only pandas, numpy, matplotlib and seaborn; no HPC or remote paths.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

DATA, IMAGES = Path("../data"), Path("../images")
IMAGES.mkdir(exist_ok=True)
plt.rcParams["svg.hashsalt"] = "genome_entropy"   # deterministic SVG ids if SVG is ever written

# Colour slots (fixed order, colour-blind-checked categorical palette used across the LOGAN figures)
SLOTS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7"]
INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e4e3df", "#fcfcfb"
sns.set_theme(context="paper", style="ticks", font_scale=1.15, rc={
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.facecolor": SURFACE, "figure.facecolor": "white", "grid.color": GRID,
    "axes.grid": True, "grid.linewidth": 0.6, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "lines.linewidth": 2, "font.family": "DejaVu Sans"})


def panel(ax, letter, x=-0.08, y=1.04):
    ax.text(x, y, letter, transform=ax.transAxes, fontsize=15, fontweight="bold", va="bottom", ha="right")


def save(fig, name):
    """Vector PDF (fixed metadata, so reruns are identical) and a 300-dpi PNG preview."""
    fig.savefig(IMAGES / f"{name}.pdf", bbox_inches="tight", metadata={"CreationDate": None, "ModDate": None})
    fig.savefig(IMAGES / f"{name}.png", dpi=300, bbox_inches="tight", metadata={"Software": None})
    plt.close(fig)
    print("wrote", IMAGES / f"{name}.pdf", IMAGES / f"{name}.png")

In [ ]:
L = pd.read_csv(DATA / "figL07_evidence_by_length.tsv.gz", sep="\t")
E = pd.read_csv(DATA / "figL07_evidence_classes.tsv.gz", sep="\t", index_col=0)
B = pd.read_csv(DATA / "figL07_hit_biome.tsv.gz", sep="\t")
ARMS = ["prodigal", "candidate", "intergenic_lo", "antisense"]
LABEL = {"prodigal": "complete Prodigal genes", "candidate": "candidates", "intergenic_lo": "low-score intergenic",
         "antisense": "antisense shadows"}
COLOR = dict(zip(ARMS, [SLOTS[0], SLOTS[1], SLOTS[2], SLOTS[3]]))
EV = ["sequence+structure", "structure-only", "sequence-only", "neither"]
EVC = dict(zip(EV, [SLOTS[0], SLOTS[1], SLOTS[3], GRID]))
SK = ["Bacteria", "Archaea", "Eukaryota", "Viruses"]
SKC = dict(zip(SK, [SLOTS[0], SLOTS[6], SLOTS[2], SLOTS[1]]))

fig = plt.figure(figsize=(18, 11))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.25], hspace=0.38, wspace=0.42)
ax = fig.add_subplot(gs[0, 0])
L["lo"] = L.length_bin.str.extract(r"\[(\d+)").astype(float)
for a in ARMS:
    g = L[L.arm == a].sort_values("lo")
    lab = g.length_bin.str.replace(r"\[(\d+)\.0, (\d+)\.0\)", r"\1–\2", regex=True)
    ax.plot(lab, g.fullcov_struct_rate, marker="o", ms=8, color=COLOR[a], label=LABEL[a])
ax.set_ylim(0, 1)
ax.set_xlabel("ORF length (aa)")
ax.set_ylabel("full-length structural hit rate")
ax.legend(loc="upper left", fontsize=9)
panel(ax, "a")

ax = fig.add_subplot(gs[0, 1])
Ee = E.reindex(ARMS)[EV]
left = np.zeros(len(ARMS))
for ev in EV:
    ax.barh(range(len(ARMS)), Ee[ev], left=left, color=EVC[ev], label=ev, height=0.6, edgecolor="white", lw=2)
    left += Ee[ev].to_numpy()
ax.set_yticks(range(len(ARMS)), [LABEL[a] for a in ARMS])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("fraction of queries (full-length evidence, five databases)")
ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.4, 1.0), fontsize=9)
panel(ax, "b", x=-0.3)

Bc = B[B.arm == "candidate"].set_index("biome").sort_values("rate")
Bp = B[B.arm == "prodigal"].set_index("biome").reindex(Bc.index)
y = np.arange(len(Bc))
ax = fig.add_subplot(gs[1, 0])
ax.barh(y - 0.19, Bp.rate, height=0.36, color=SLOTS[0], label="complete Prodigal genes")
ax.barh(y + 0.19, Bc.rate, height=0.36, color=SLOTS[1], label="candidates")
ax.set_yticks(y, [b.replace("_", " ") for b in Bc.index])
ax.set_xlim(0, 1)
ax.set_xlabel("full-length structural hit rate")
ax.legend(ncol=2, loc="lower center", bbox_to_anchor=(0.5, 1.0), fontsize=9)
panel(ax, "c", x=-0.25)
ax = fig.add_subplot(gs[1, 1])
left = np.zeros(len(Bc))
for k in SK:
    v = Bc[f"{k}_of_hits"].fillna(0).to_numpy()
    ax.barh(y, v, left=left, color=SKC[k], label=k, height=0.6, edgecolor="white", lw=2)
    left += v
ax.set_yticks(y, [b.replace("_", " ") for b in Bc.index])
ax.set_xlim(0, 1)
ax.set_xlabel("candidates with a hit: superkingdom of best target")
ax.legend(ncol=4, loc="lower center", bbox_to_anchor=(0.5, 1.0), fontsize=9)
panel(ax, "d", x=-0.25)
save(fig, "figL07_candidate_evidence_detail")